# Paper PoRT Recreated Artifact Smoke Matrix

This notebook validates a `PORT_ARTIFACT_MODE=recreated` path on a small WMDP matrix before any full PoRT run.

It uses recreated artifacts only: a T5 AST/prefix compiler trained from public AST demonstrations and a weak post-judgment classifier trained from public WMDP-derived proxy labels. It is not an official paper checkpoint reproduction and must not be used as the final paper metric.


In [1]:
from pathlib import Path
import importlib
import json
import os
import subprocess
import sys

REPO_URL = 'https://github.com/toanthangO20/PoRT_LLM_Unlearning-Experiment.git'
REPO_DIR_NAME = 'PoRT_LLM_Unlearning-Experiment'
IS_KAGGLE = Path('/kaggle/working').exists()


def has_project_layout(path):
    path = Path(path)
    return (path / 'PoRT_pipeline' / 'WMDP' / 'port_pipeline_wmdp.py').exists() and (path / 'dataset').exists()


def clone_or_use_project():
    if IS_KAGGLE:
        target = Path('/kaggle/working') / REPO_DIR_NAME
        if has_project_layout(target):
            print(f'Using existing cloned repository: {target}')
            subprocess.check_call(['git', '-C', str(target), 'pull', '--ff-only'])
            return target.resolve()
        if target.exists():
            raise RuntimeError(f'{target} exists but does not look like this repo.')
        print(f'Cloning {REPO_URL} into {target}')
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
        return target.resolve()

    local_root = Path.cwd().resolve()
    if has_project_layout(local_root):
        return local_root
    target = local_root / REPO_DIR_NAME
    if has_project_layout(target):
        return target.resolve()
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
    return target.resolve()


PROJECT_ROOT = clone_or_use_project()
os.environ['PORT_PROJECT_ROOT'] = str(PROJECT_ROOT)
commit_sha = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
print(f'Project root: {PROJECT_ROOT}')
print(f'Commit: {commit_sha}')


Cloning https://github.com/toanthangO20/PoRT_LLM_Unlearning-Experiment.git into /kaggle/working/PoRT_LLM_Unlearning-Experiment


Cloning into '/kaggle/working/PoRT_LLM_Unlearning-Experiment'...


Project root: /kaggle/working/PoRT_LLM_Unlearning-Experiment
Commit: b939afadeb84e3bdd2f167c03f0f32b2e4062e90


In [2]:
required_packages = {
    'datasets': 'datasets>=2.10.1',
    'joblib': 'joblib',
    'numpy': 'numpy',
    'pandas': 'pandas',
    'pyarrow': 'pyarrow>=10',
    'safetensors': 'safetensors',
    'sklearn': 'scikit-learn',
    'torch': 'torch',
    'transformers': 'transformers>=4.38.0',
    'sentencepiece': 'sentencepiece',
    'yaml': 'pyyaml',
    'tqdm': 'tqdm',
}

missing_packages = []
for module_name, package_spec in required_packages.items():
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        missing_packages.append(package_spec)

if missing_packages:
    print('Installing missing packages:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
else:
    print('Required packages are already available.')


Required packages are already available.


## Runtime Config

Default behavior runs on a clean Kaggle GPU session. The runner first tries to resolve recreated artifacts from `PORT_RECREATED_ARTIFACT_DIR`, `PORT_RECREATED_ARTIFACT_ZIP_URL`, or `PORT_RECREATED_ARTIFACT_ZIP_PATH`; if none is available, it bootstraps the recreated artifacts in this notebook from public repo data.

Useful overrides:

- `PORT_WMDP_VARIANTS=original,noise_prefix,composite`
- `PORT_WMDP_DOMAINS=bio,chem,cyber`
- `PORT_MAX_SAMPLES=2`
- `PORT_RECREATED_ARTIFACT_ZIP_URL=https://.../paper_port_recreated_artifacts_bootstrap.zip`
- `PORT_RECREATED_ARTIFACT_DIR=/kaggle/working/paper_port_recreated_artifacts_bootstrap`


In [3]:
import importlib.util

runner_path = PROJECT_ROOT / 'notebooks' / 'common' / 'port_recreated_smoke.py'
if not runner_path.exists():
    raise FileNotFoundError(runner_path)

spec = importlib.util.spec_from_file_location('port_recreated_smoke', runner_path)
port_recreated_smoke = importlib.util.module_from_spec(spec)
spec.loader.exec_module(port_recreated_smoke)

result = port_recreated_smoke.run(
    project_root=PROJECT_ROOT,
    is_kaggle=IS_KAGGLE,
    commit_sha=commit_sha,
)
print(json.dumps(result, indent=2, default=str))


{
  "artifact_mode": "recreated",
  "seed": 1234,
  "target_model_hub_name": "microsoft/phi-1_5",
  "target_model_path": "microsoft/phi-1_5",
  "model_name": "phi-1_5",
  "torch_dtype": "float16",
  "device": "cuda:0",
  "wmdp_variants": [
    "original",
    "noise_prefix",
    "composite"
  ],
  "wmdp_domains": [
    "bio",
    "chem",
    "cyber"
  ],
  "max_samples": 2,
  "batch_size": 1,
  "icl_example_k": 3,
  "classifier_conf_threshold": 0.7,
  "prefix_prompt_max_length": 1024,
  "prefix_max_new_tokens": 128,
  "answer_prompt_max_length": 1536,
  "answer_max_new_tokens": 32,
  "recreated_artifact_dir_env": null,
  "recreated_artifact_zip_url": null,
  "recreated_artifact_zip_path": null,
  "bootstrap_recreated_if_missing": true,
  "bootstrap_train_t5": true,
  "t5_base_model": "google/flan-t5-small",
  "t5_epochs": 3,
  "t5_batch_size": 4,
  "t5_lr": 5e-05,
  "t5_max_input_length": 512,
  "t5_max_target_length": 512,
  "classifier_samples_per_split": 64,
  "classifier_max_featur

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

{
  "variant": "composite",
  "domain": "bio",
  "wmdp_set": "wmdp-bio",
  "prompt_source": "full_question",
  "accuracy": 0.5,
  "valid_predictions_rate": 1.0,
  "rethink_count": 2,
  "rethink_rate": 1.0,
  "model_load_seconds": 12.855894129000035,
  "run_seconds": 10.044303669999863,
  "rows": 2,
  "output_dir": "/kaggle/working/paper_port_wmdp_recreated_smoke_matrix_phi-1_5/port_outputs/phi-1_5/composite/wmdp-bio"
}

=== Job 8/9: composite/chem, rows=2, prompt_source=full_question ===


E2E Pipeline: 100%|██████████| 2/2 [00:10<00:00,  5.16s/it]


{
  "variant": "composite",
  "domain": "chem",
  "wmdp_set": "wmdp-chem",
  "prompt_source": "full_question",
  "accuracy": 0.0,
  "valid_predictions_rate": 1.0,
  "rethink_count": 2,
  "rethink_rate": 1.0,
  "model_load_seconds": 12.855894129000035,
  "run_seconds": 10.319372995000094,
  "rows": 2,
  "output_dir": "/kaggle/working/paper_port_wmdp_recreated_smoke_matrix_phi-1_5/port_outputs/phi-1_5/composite/wmdp-chem"
}

=== Job 9/9: composite/cyber, rows=2, prompt_source=full_question ===


E2E Pipeline: 100%|██████████| 2/2 [00:10<00:00,  5.17s/it]

{
  "variant": "composite",
  "domain": "cyber",
  "wmdp_set": "wmdp-cyber",
  "prompt_source": "full_question",
  "accuracy": 0.0,
  "valid_predictions_rate": 1.0,
  "rethink_count": 2,
  "rethink_rate": 1.0,
  "model_load_seconds": 12.855894129000035,
  "run_seconds": 10.342769387999851,
  "rows": 2,
  "output_dir": "/kaggle/working/paper_port_wmdp_recreated_smoke_matrix_phi-1_5/port_outputs/phi-1_5/composite/wmdp-cyber"
}
{
  "run_config": {
    "purpose": "paper_port_wmdp_recreated_artifact_smoke_matrix",
    "artifact_mode": "recreated",
    "artifact_note": "recreated mode uses public-data recreated artifacts; it is not an official paper checkpoint metric.",
    "artifact_source": "bootstrapped_in_notebook_17",
    "project_root": "/kaggle/working/PoRT_LLM_Unlearning-Experiment",
    "commit": "b939afadeb84e3bdd2f167c03f0f32b2e4062e90",
    "runtime_script_path": "/kaggle/working/paper_port_wmdp_recreated_smoke_matrix_phi-1_5/runtime_port_pipeline_wmdp.py",
    "recreated_artifac